<h1>Fine-Tuning SAM2 Image Predictor w/ CamVid</h1>

Credits:
- [Video](https://www.youtube.com/watch?v=bcwLbmALyLI)
- [Medium Article](https://medium.com/towards-data-science/train-fine-tune-segment-anything-2-sam-2-in-60-lines-of-code-928dd29a63b3)
- [Repository](https://github.com/sagieppel/fine-tune-train_segment_anything_2_in_60_lines_of_code/tree/main)

In [ ]:
import numpy as np
import torch
import cv2
import os
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
import matplotlib.pyplot as plt

In [ ]:
data_dir = "../CamVid/"
data = []

# ff=index, name=filename
for ff, name in enumerate(os.listdir(data_dir + "train/")):
    try:
        data.append({
            "image":data_dir + "train/"+name,
            "annotation":data_dir+"train_labels/"+name[:-4]+"_L.png"
        })
    except:
        print("Encountered error with", name, ".")

for pair in data[:5]:
    print(pair["image"], pair["annotation"]) # to test above works

In [ ]:
def disp_np_array_as_img(img:np.array):
    plt.figure(figsize=(10, 10))
    plt.imshow(img)    
    plt.axis('off')
    plt.show()

In [ ]:
def read_batch(data): 
    # read random image and its annotatio from the CamVid dataset

    entry = data[np.random.randint(len(data))] # Choose a random entry
    img = cv2.imread(entry["image"])
    ann_map = cv2.imread(entry["annotation"])

    # Normally we'd resize here. However, images are smaller than 1024 in both dimensions.
    # Thus, we skip resizing.
    all_pixels = ann_map.reshape(-1, 3)
    unique_colors = np.unique(all_pixels, axis=0)
    unique_colors = unique_colors = unique_colors[~np.all(unique_colors == [0,0,0], axis=1)]
    
    # Using the annotation, get all unique colors. Then, sort them into binary masks for each color.
    points = []
    masks = []

    for color in unique_colors:
        binary_mask = np.all(ann_map == color, axis=2).astype(np.uint8) # make binary mask
        # print(binary_mask.shape)
        # print(np.unique(binary_mask))
        mask = np.zeros(shape=(720, 960, 3), dtype=np.uint8)

        # Below binary mask isn't setting??
        mask[binary_mask == 1] = color
        # print(mask[binary_mask == 1].shape)
        masks.append(mask)
        coords = np.argwhere(mask > 0)
        yx = np.array(coords[np.random.randint(len(coords))]) # choose random point/coordinate from mask
        points.append([[yx[1], yx[0]]]) # x,y
        
    return img, np.array(masks), np.array(points), np.ones([len(masks), 1])

if False: read_batch(data) # testing code

In [ ]:
sam2_checkpoint = "./checkpoints/sam2.1_hiera_small.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_s.yaml"

sam2_model = build_sam2(model_cfg, sam2_checkpoint, device="cuda") # load mmodel
predictor = SAM2ImagePredictor(sam2_model) # load net

base_sam2 = build_sam2(model_cfg, sam2_checkpoint, device="cuda") # load mmodel
base = SAM2ImagePredictor(base_sam2) # base sam2

In [ ]:
predictor.model.sam_mask_decoder.train(True) # enable training of mask decoder
predictor.model.sam_prompt_encoder.train(True) # enable training of prompt decoder

In [ ]:
optimizer = torch.optim.AdamW(
    params=predictor.model.parameters(), 
    lr=1e-5,
    weight_decay=4e-5
)
scaler = torch.cuda.amp.GradScaler()

In [ ]:
def read_image(image_path, mask_path):
    # read an image and its mask
    image = cv2.imread(image_path)[...,::-1] # convert bgr to rgb
    mask = cv2.imread(mask_path, 0) # load masks in grayscale

    # Again, no resizing since images are less than 1024px on both dimensions

    return image, mask

def get_points(mask, num_points): # Sample points inside the input mask
    points = []
    for i in range(num_points):
        coords = np.argwhere(mask > 0)
        yx = np.array(coords[np.random.randint(len(coords))])
        points.append([[yx[1], yx[0]]])
    return np.array(points)

data_dir = "../CamVid/"
val = []
for ff, name in enumerate(os.listdir(data_dir + "val/")[:5]):
    try:
        val.append({
            "image":data_dir + "val/"+name,
            "annotation":data_dir+"val_labels/"+name[:-4]+"_L.png"
        })
    except:
        print("Encountered error with", name, ".")


for pair in val:
    print(pair["image"], pair["annotation"]) # to test above works

### TODO: Change below color selection to feed map segmentation into YOLO, get an obj, and apply the appropriate CamVid coloring.
Refer to ```sam2-yolo-pipeline-with-noise.ipynb``` for help.

In [ ]:
%pip install ultralytics

In [ ]:
import pandas as pd

table = pd.read_csv("../download_dataset/class_dict.csv")
# print(table.head())

# dict based on english labels
class_dict = {row["name"].lower() : [row["r"] / 255, row["g"] / 255, row["b"] / 255] for _, row in table.iterrows()}
color_labels = [class_dict[key] for key in class_dict]

print("Class dict:")
print(class_dict)
label_map = {
    'person':'pedestrian'
}

def label_to_color(label:str):
    if label in class_dict:
            color = np.concatenate([np.array(class_dict[label]), np.array([1])], axis=0)
    else:
        if label in label_map:
            label = label_map[label]
            if label in class_dict:
                color = np.concatenate([np.array(class_dict[label]), np.array([1])], axis=0)
            else:
                print("Could not find", label)
                color = np.concatenate([np.random.random(3), np.array([1])], axis=0)
        else:
            print("Could not find", label)
            color = np.concatenate([np.random.random(3), np.array([1])], axis=0)
    return color

In [ ]:
from ultralytics import YOLO
yolo_model = YOLO('yolov8n.pt')

In [ ]:
# predict masks
def test_predictor(predictor):   
    pair = val[np.random.randint(low=0, high=len(val))] # update later to for-loop some images

    image_path, mask_path = pair["image"], pair["annotation"]
    image, mask = read_image(image_path, mask_path)
    gt = mask
    input_points = get_points(mask, num_points=30) # arbitrarily get 30 points

    with torch.no_grad(): # prevent the net from caclulate gradient (more efficient inference)
        predictor.set_image(image.copy()) # image encoder
        masks, scores, logits = predictor.predict(  # prompt encoder + mask decoder
            point_coords=input_points,
            point_labels=np.ones([input_points.shape[0],1])
        )

        masks=masks[:,0].astype(bool)
        shorted_masks = masks[np.argsort(scores[:,0])][::-1].astype(bool)

        seg_map = np.zeros_like(shorted_masks[0],dtype=np.uint8)
        occupancy_mask = np.zeros_like(shorted_masks[0],dtype=bool)

        for i in range(shorted_masks.shape[0]):
            mask = shorted_masks[i]
            if (mask*occupancy_mask).sum()/mask.sum()>0.15: continue 
            mask[occupancy_mask]=0
            seg_map[mask]=i+1
            occupancy_mask[mask]=1

        rgb_image = np.zeros((seg_map.shape[0], seg_map.shape[1], 3), dtype=np.uint8)
        for id_class in range(1,seg_map.max()+1):
            rgb_image[seg_map == id_class] = [np.random.randint(255), np.random.randint(255), np.random.randint(255)]

        print("Predicted")
        plt.imshow(rgb_image) # segmented
        plt.axis('off')
        plt.show()

        print("Mix")
        plt.imshow((rgb_image/2+image/2).astype(np.uint8)) # mix
        plt.axis('off')
        plt.show()

        print("Original")
        plt.imshow(image) # original image
        plt.axis('off')
        plt.show()

        print("Ground Truth")
        plt.imshow(gt) # original image
        plt.axis('off')
        plt.show()

    return rgb_image


In [ ]:
def get_uniq_pixels(ann_map):
    all_pixels = ann_map.reshape(-1, 3)
    unique_colors = np.unique(all_pixels, axis=0)
    return unique_colors

### Debug code

In [ ]:
if False: # change to true if debugging as this code kills kernel
    sample=1

    image, mask, input_point, input_label = read_batch(data) # test result
    print("read_batch shapes:", image.shape, mask.shape, input_point.shape, input_label.shape)

    predictor.set_image(image) # apply SAM2 image encoder to training image

    # prompt encoding
    mask_input, unnorm_coords, labels, unnorm_box = predictor._prep_prompts(input_point, input_label, box=None, mask_logits=None, normalize_coords=True)
    sparse_embeddings, dense_embeddings = predictor.model.sam_prompt_encoder(points=(unnorm_coords, labels), boxes=None, masks=None)

    # mask decoder
    batched_mode = unnorm_coords.shape[0] > 1 # multi object prediction
    high_res_features = [feat_level[-1].unsqueeze(0) for feat_level in predictor._features["high_res_feats"]]
    low_res_masks, prd_scores, _, _ = predictor.model.sam_mask_decoder(image_embeddings=predictor._features["image_embed"][-1].unsqueeze(0),image_pe=predictor.model.sam_prompt_encoder.get_dense_pe(),sparse_prompt_embeddings=sparse_embeddings,dense_prompt_embeddings=dense_embeddings,multimask_output=True,repeat_image=batched_mode,high_res_features=high_res_features,)
    prd_masks = predictor._transforms.postprocess_masks(low_res_masks, predictor._orig_hw[-1])# Upscale the masks to the original image resolution

    print("predicted mask shape", prd_masks.shape)
    print("gt (ground truth) mask shape", mask.shape)
    
    print("shape of predicted scores", prd_scores.shape) # Could these show uncertainty? confidence score == uncertainty score?
                                                        # Dimension of 3 - one for low-grain, one for high-grain, one for middle
                                                        # SAM2 automatically chooses the best mask
    print(prd_scores[0], "\n", prd_scores[-1])

    pair = val[np.random.randint(low=0, high=len(val))] # update later to for-loop some images

    image_path, mask_path = pair["image"], pair["annotation"]
    image, mask = read_image(image_path, mask_path)
    gt = mask
    input_points = get_points(mask, num_points=30) # arbitrarily get 30 points

    with torch.no_grad(): # prevent the net from caclulate gradient (more efficient inference)
        predictor.set_image(image.copy()) # image encoder
        masks, scores, logits = predictor.predict(  # prompt encoder + mask decoder
            point_coords=input_points,
            point_labels=np.ones([input_points.shape[0],1])
        )

        masks=masks[:,0].astype(bool)
        shorted_masks = masks[np.argsort(scores[:,0])][::-1].astype(bool)

        seg_map = np.zeros_like(shorted_masks[0],dtype=np.uint8)
        occupancy_mask = np.zeros_like(shorted_masks[0],dtype=bool)

        for i in range(shorted_masks.shape[0]):
            mask = shorted_masks[i]
            if (mask*occupancy_mask).sum()/mask.sum()>0.15: continue 
            mask[occupancy_mask]=0
            seg_map[mask]=i+1
            occupancy_mask[mask]=1

        rgb_image = np.zeros((seg_map.shape[0], seg_map.shape[1], 3), dtype=np.uint8)
        print("base", rgb_image.shape)
        for id_class in range(1,seg_map.max()+1):
        
            rgb_image[seg_map == id_class] = [np.random.randint(255), np.random.randint(255), np.random.randint(255)]

            results = yolo_model.predict(source=rgb_image, conf=0.25)
            try:
                print("found:", results.boxes.cls.tolist())
            except:
                print("no objects detected.")
            
        print("Predicted")
        plt.imshow(rgb_image) # segmented
        plt.axis('off')
        plt.show()

        print("Mix")
        plt.imshow((rgb_image/2+image/2).astype(np.uint8)) # mix
        plt.axis('off')
        plt.show()

        print("Original")
        plt.imshow(image) # original image
        plt.axis('off')
        plt.show()

        print("Ground Truth")
        plt.imshow(gt) # original image
        plt.axis('off')
        plt.show()

In [ ]:
# training loop
from datetime import datetime

sample=1
num_steps = 6000 # 1 000 000
# num_steps = 10

for itr in range(1, int(num_steps) + 1):
    with torch.cuda.amp.autocast(): # cast to mix precision

        image, mask, input_point, input_label = read_batch(data)
        if mask.shape[0] == 0: continue # skip empty batches

        # ---------------------------------------- FINETUNED SAM2 PREDICTION ----------------------------------------
        predictor.set_image(image) # apply SAM2 image encoder to training image

        # prompt encoding
        mask_input, unnorm_coords, labels, unnorm_box = predictor._prep_prompts(input_point, input_label, box=None, mask_logits=None, normalize_coords=True)
        sparse_embeddings, dense_embeddings = predictor.model.sam_prompt_encoder(points=(unnorm_coords, labels), boxes=None, masks=None)

        # mask decoder
        batched_mode = unnorm_coords.shape[0] > 1 # multi object prediction
        high_res_features = [feat_level[-1].unsqueeze(0) for feat_level in predictor._features["high_res_feats"]]

        low_res_masks, prd_scores, _, _ = predictor.model.sam_mask_decoder(
            image_embeddings=predictor._features["image_embed"][-1].unsqueeze(0),
            image_pe=predictor.model.sam_prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_embeddings,
            dense_prompt_embeddings=dense_embeddings,
            multimask_output=True,
            repeat_image=batched_mode,
            high_res_features=high_res_features,)
        
        prd_masks = predictor._transforms.postprocess_masks(low_res_masks, predictor._orig_hw[-1])# Upscale the masks to the original image resolution

        # print(prd_masks.shape, mask.shape)
        prd_masks = prd_masks.permute(0,2,3,1)

        # ---------------------------------------- BASE SAM2 PREDICTION ----------------------------------------

        base.set_image(image) # apply SAM2 image encoder to training image

        # prompt encoding
        mask_input, unnorm_coords, labels, unnorm_box = base._prep_prompts(input_point, input_label, box=None, mask_logits=None, normalize_coords=True)
        sparse_embeddings, dense_embeddings = predictor.model.sam_prompt_encoder(points=(unnorm_coords, labels), boxes=None, masks=None)

        # mask decoder
        batched_mode = unnorm_coords.shape[0] > 1 # multi object prediction
        high_res_features = [feat_level[-1].unsqueeze(0) for feat_level in base._features["high_res_feats"]]

        low_res_masks, prd_scores, _, _ = base.model.sam_mask_decoder(
            image_embeddings=base._features["image_embed"][-1].unsqueeze(0),
            image_pe=base.model.sam_prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_embeddings,
            dense_prompt_embeddings=dense_embeddings,
            multimask_output=True,
            repeat_image=batched_mode,
            high_res_features=high_res_features,)
        
        base_masks = base._transforms.postprocess_masks(low_res_masks, base._orig_hw[-1])# Upscale the masks to the original image resolution

        # print(prd_masks.shape, mask.shape)
        base_masks = base_masks.permute(0,2,3,1)

        # ---------------------------------------- LOSS CALCULATION ----------------------------------------

        # segmentation loss calculation
        gt_mask = torch.tensor(mask.astype(np.float32)).cuda()
        
        prd_mask = torch.sigmoid(prd_masks) # Turn logit map to probability map

        seg_loss = (-gt_mask * torch.log(prd_mask + 0.00001) - (1 - gt_mask) * torch.log((1 - prd_mask) + 0.00001)).mean() # cross entropy loss

        mse = torch.mean((prd_masks - base_masks) ** 2)

        loss = seg_loss + mse

        # backpropogate loss
        predictor.model.zero_grad() # empty gradient        
        scaler.scale(loss).backward()  # Backpropogate - only requirement is that it is a tensor with a gradient
        
        scaler.step(optimizer)
        scaler.update() # Mix precision

        if itr%10==0:
            test_predictor(predictor)
            print(f"Confidence Scores={prd_scores.cpu().detach().numpy()}")

        # Save model
        if itr%2000==0: 
            date_str = str(datetime.now()).replace(":","-")
            torch.save(predictor.model.state_dict(), f"../models/model-{date_str}.torch")
            print("saved model.")
    
        # Display results
        # if itr==1: mean_iou=0
        # detached_iou=np.mean(iou.cpu().detach().numpy())
        # mean_iou = mean_iou * 0.99 + 0.01 * detached_iou

        scaler_seg = loss.cpu().detach()
        print(f"step {itr}) | Loss={scaler_seg} | MSE between base and finetuned={mse}")

### Testing on some Images

In [ ]:
import numpy as np
import torch
import cv2
import os
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
import matplotlib.pyplot as plt

# Validate our finetuned SAM2

# use bfloat16 for memory efficiency
torch.autocast(device_type="cuda", dtype=torch.float32).__enter__()

# load some example images
data_dir = "../CamVid/"
val = []
for ff, name in enumerate(os.listdir(data_dir + "val/")[:5]):
    try:
        val.append({
            "image":data_dir + "val/"+name,
            "annotation":data_dir+"val_labels/"+name[:-4]+"_L.png"
        })
    except:
        print("Encountered error with", name, ".")


for pair in val:
    print(pair["image"], pair["annotation"]) # to test above works

In [ ]:
def read_image(image_path, mask_path):
    # read an image and its mask
    image = cv2.imread(image_path)[...,::-1] # convert bgr to rgb
    mask = cv2.imread(mask_path, 0) # load masks in grayscale

    # Again, no resizing since images are less than 1024px on both dimensions

    return image, mask

def get_points(mask, num_points, random=True): # Sample points inside the input mask
    points = []
    coords = np.argwhere(mask > 0)
    if random:
        for i in range(num_points):
            yx = np.array(coords[np.random.randint(len(coords))])
            points.append([[yx[1], yx[0]]])
    else:
        n = len(coords)
        stride = max(n // num_points, 1)
        for i in range(0, n, stride):
            yx = coords[i]
            points.append([[yx[1], yx[0]]])
    return np.array(points)

data_dir = "../CamVid/"
val = []
for ff, name in enumerate(os.listdir(data_dir + "val/")[:5]):
    try:
        val.append({
            "image":data_dir + "val/"+name,
            "annotation":data_dir+"val_labels/"+name[:-4]+"_L.png"
        })
    except:
        print("Encountered error with", name, ".")


for pair in val:
    print(pair["image"], pair["annotation"]) # to test above works

In [ ]:
pair = val[0] # update later to for-loop some images

image_path, mask_path = pair["image"], pair["annotation"]
image, mask = read_image(image_path, mask_path)
input_points = get_points(mask, num_points=30) # arbitrarily get 30 points

In [ ]:
model_dir = "../models/"

sam2_checkpoint = "./checkpoints/sam2.1_hiera_small.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_s.yaml"
sam2_model = build_sam2(model_cfg, sam2_checkpoint, device="cuda")

# build finetuned model and load weights
predictor = SAM2ImagePredictor(sam2_model)
predictor.model.load_state_dict(torch.load(model_dir + os.listdir(model_dir)[-1]))
print("Loaded model:", model_dir + os.listdir(model_dir)[-1])

base = SAM2ImagePredictor(sam2_model)

In [ ]:
def show_mask(mask, ax, random_color=False, borders = True):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30/255, 144/255, 255/255, 0.6])
    h, w = mask.shape[-2:]
    mask = mask.astype(np.uint8)
    mask_image =  mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    if borders:
        import cv2
        contours, _ = cv2.findContours(mask,cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE) 
        # Try to smooth contours
        contours = [cv2.approxPolyDP(contour, epsilon=0.01, closed=True) for contour in contours]
        mask_image = cv2.drawContours(mask_image, contours, -1, (1, 1, 1, 0.5), thickness=2) 
    ax.imshow(mask_image)

def show_points(coords, labels, ax, marker_size=375):
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)   

def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor=(0, 0, 0, 0), lw=2))    

def show_masks(image, masks, scores, point_coords=None, box_coords=None, input_labels=None, borders=True):
    for i, (mask, score) in enumerate(zip(masks, scores)):
        plt.figure(figsize=(10, 10))
        plt.imshow(image)
        show_mask(mask, plt.gca(), borders=borders)
        if point_coords is not None:
            assert input_labels is not None
            show_points(point_coords, input_labels, plt.gca())
        if box_coords is not None:
            # boxes
            show_box(box_coords, plt.gca())
        if len(scores) > 1:
            plt.title(f"Mask {i+1}, Score: {score:.3f}", fontsize=18)
        plt.axis('off')
        plt.show()

Finetuned SAM2

In [ ]:
import numpy as np
import torch
import cv2
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

# use bfloat16 for the entire script (memory efficient)
torch.autocast(device_type="cuda", dtype=torch.bfloat16).__enter__()

In [ ]:
pair = val[0] # update later to for-loop some images

image_path, mask_path = pair["image"], pair["annotation"]

def read_image(image_path, mask_path): # read and resize image and mask
    img = cv2.imread(image_path)[...,::-1]  # read image as rgb
    mask = cv2.imread(mask_path,0) # mask of the region we want to segment

    return img, mask

image, mask = read_image(image_path, mask_path)
input_points = get_points(mask, num_points=30) # arbitrarily get 30 points

sam2_checkpoint = "./checkpoints/sam2.1_hiera_small.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_s.yaml"
sam2_model = build_sam2(model_cfg, sam2_checkpoint, device="cuda")
predictor = SAM2ImagePredictor(sam2_model)
predictor.model.load_state_dict(torch.load(model_dir + os.listdir(model_dir)[-1]))
print("Loaded model:", model_dir + os.listdir(model_dir)[-1])

In [ ]:
with torch.no_grad(): # prevent the net from caclulate gradient (more efficient inference)
    predictor.set_image(image.copy()) # image encoder
    masks, scores, logits = predictor.predict(  # prompt encoder + mask decoder
        point_coords=input_points,
        point_labels=np.ones([input_points.shape[0],1])
    )

    masks=masks[:,0].astype(bool)
    shorted_masks = masks[np.argsort(scores[:,0])][::-1].astype(bool)

    seg_map = np.zeros_like(shorted_masks[0],dtype=np.uint8)
    occupancy_mask = np.zeros_like(shorted_masks[0],dtype=bool)

    for i in range(shorted_masks.shape[0]):
        mask = shorted_masks[i]
        if (mask*occupancy_mask).sum()/mask.sum()>0.15: continue 
        mask[occupancy_mask]=0
        seg_map[mask]=i+1
        occupancy_mask[mask]=1

    rgb_image = np.zeros((seg_map.shape[0], seg_map.shape[1], 3), dtype=np.uint8)
    for id_class in range(1,seg_map.max()+1):
        rgb_image[seg_map == id_class] = [np.random.randint(255), np.random.randint(255), np.random.randint(255)]

    plt.imshow(rgb_image) # segmented
    plt.axis('off')
    plt.show()

    plt.imshow((rgb_image/2+image/2).astype(np.uint8)) # mix
    plt.axis('off')
    plt.show()

    plt.imshow(image) # original image
    plt.axis('off')
    plt.show()


In [ ]:
base = SAM2ImagePredictor(sam2_model)

with torch.no_grad(): # prevent the net from caclulate gradient (more efficient inference)
    base.set_image(image.copy()) # image encoder
    masks, scores, logits = base.predict(  # prompt encoder + mask decoder
        point_coords=input_points,
        point_labels=np.ones([input_points.shape[0],1])
    )

    masks=masks[:,0].astype(bool)
    shorted_masks = masks[np.argsort(scores[:,0])][::-1].astype(bool)

    seg_map = np.zeros_like(shorted_masks[0],dtype=np.uint8)
    occupancy_mask = np.zeros_like(shorted_masks[0],dtype=bool)

    for i in range(shorted_masks.shape[0]):
        mask = shorted_masks[i]
        if (mask*occupancy_mask).sum()/mask.sum()>0.15: continue 
        mask[occupancy_mask]=0
        seg_map[mask]=i+1
        occupancy_mask[mask]=1

    rgb_image = np.zeros((seg_map.shape[0], seg_map.shape[1], 3), dtype=np.uint8)
    for id_class in range(1,seg_map.max()+1):
        rgb_image[seg_map == id_class] = [np.random.randint(255), np.random.randint(255), np.random.randint(255)]

    test_predictor(base)

# SCRAP WORK BELOW - USE IF NEEDED:

In [ ]:
pair = val[0] # update later to for-loop some images

image_path, mask_path = pair["image"], pair["annotation"]
image, mask = read_image(image_path, mask_path)

predictor.set_image(image.copy()) # apply SAM2 image encoder to training image

input_points = get_points(mask, num_points=4, random=False).reshape(-1,2) # arbitrarily get 30 points
input_labels = np.ones(len(input_points))  # All positive points

# Predict mask
masks, scores, logits = predictor.predict(
    point_coords=input_points,
    point_labels=input_labels,
    multimask_output=True  # Set to True if you want multiple masks
)
# print("output shape", masks[0].shape)

plt.figure(figsize=(10,10))
plt.axis('off')
plt.imshow(masks[0])

From the SAM2 Automatic Mask Generator Notebook:

```Since SAM 2 can efficiently process prompts, masks for the entire image can be generated by sampling a large number of prompts over an image.```

```The class SAM2AutomaticMaskGenerator implements this capability. It works by sampling single-point input prompts in a grid over the image, from each of which SAM can predict multiple masks. Then, masks are filtered for quality and deduplicated using non-maximal suppression. Additional options allow for further improvement of mask quality and quantity, such as running prediction on multiple crops of the image or postprocessing masks to remove small disconnected regions and holes.```

In [ ]:
pair = val[0] # update later to for-loop some images

image_path, mask_path = pair["image"], pair["annotation"]
image, mask = read_image(image_path, mask_path)

predictor.set_image(image.copy()) # apply SAM2 image encoder to training image

input_points = get_points(mask, num_points=4, random=False).reshape(-1,2) # arbitrarily get a bunch of points - 4000 seems to be a good number
input_labels = np.ones(len(input_points))  # All positive points

# Predict mask
masks, scores, logits = predictor.predict(
    point_coords=input_points,
    point_labels=input_labels,
    multimask_output=False  # Set to True if you want multiple masks
)
# print("output shape", masks[0].shape)

plt.figure(figsize=(10,10))
plt.axis('off')
plt.imshow(masks[0])

Base SAM2


In [ ]:
pair = val[0] # update later to for-loop some images

image_path, mask_path = pair["image"], pair["annotation"]
image, mask = read_image(image_path, mask_path)

base.set_image(image.copy()) # apply SAM2 image encoder to training image

input_points = get_points(mask, num_points=4, random=False).reshape(-1,2) # arbitrarily get 30 points
input_labels = np.ones(len(input_points))  # All positive points

# Predict mask
masks, scores, logits = base.predict(
    point_coords=input_points,
    point_labels=input_labels,
    multimask_output=True  # Set to True if you want multiple masks
)
print("output shape", masks[0].shape)

plt.figure(figsize=(10,10))
plt.axis('off')
plt.imshow(masks[0])